# Seeds Wheat Variety Classification

Wheat kernels from three varieties look alike to the eye, but seven geometric measurements of each kernel — area, perimeter, compactness, length, width, asymmetry and groove length, measured from soft X-ray images — separate them surprisingly well. Predict the variety from those seven numbers alone.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Seeds (id 236), 210 kernels. Note: UCI does NOT serve a data.csv for
# this dataset (404); its file is seeds_dataset.txt, served from the
# classic UCI path below. Columns are tab-separated, no header.
cols = ['area', 'perimeter', 'compactness', 'length', 'width', 'asymmetry', 'groove', 'class']
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt')),
                 sep=r'\s+', header=None, names=cols)
print(df.shape)
print('Kernels per variety code (1=Kama, 2=Rosa, 3=Canadian):')
print(df['class'].value_counts().sort_index())
df.head(3)


(210, 8)
Kernels per variety code (1=Kama, 2=Rosa, 3=Canadian):
class
1    70
2    70
3    70
Name: count, dtype: int64


,area,perimeter,compactness,length,width,asymmetry,groove,class
0,15.26,14.84,0.8710,5.763,3.312,2.221,5.220,1
1,14.88,14.57,0.8811,5.554,3.333,1.018,4.956,1
2,14.29,14.09,0.9050,5.291,3.337,2.699,4.825,1


In [2]:

feats = ['area', 'perimeter', 'compactness', 'length', 'width', 'asymmetry', 'groove']
variety = {1: 'Kama', 2: 'Rosa', 3: 'Canadian'}
X = df[feats]; y = df['class'].map(variety)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.8491
Logistic Regression accuracy: 0.9434
              precision    recall  f1-score   support

    Canadian       0.74      0.82      0.78        17
        Kama       0.81      0.72      0.76        18
        Rosa       1.00      1.00      1.00        18

    accuracy                           0.85        53
   macro avg       0.85      0.85      0.85        53
weighted avg       0.85      0.85      0.85        53



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which kernel geometry separates the varieties?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


area           0.232606
perimeter      0.216183
groove         0.201519
width          0.137560
length         0.121169
asymmetry      0.048805
compactness    0.042159
dtype: float64


Winner: Logistic Regression
